In [5]:
from datasets import load_dataset

ds = load_dataset("jg-eno/msmarco-v5.1-Qwen-Embeddings",split='train' ,streaming=True)

In [7]:
record_1 = next(iter(ds))

In [8]:
record_1.keys()

dict_keys(['token_embeddings', 'sentence_embeddings', 'input_ids', 'attention_mask', 'seq_lengths', 'texts'])

In [19]:
len(record_1['token_embeddings']) // 1024

110

In [22]:
def chunk_list(lst, n):
    return [lst[i:i+n] for i in range(0, len(lst),n)]

l = [1,2,3,4,5,6]
chunk_list(l,2)

[[1, 2], [3, 4], [5, 6]]

In [27]:
tok_emb = chunk_list(record_1['token_embeddings'],1024)

In [28]:
import numpy as np
tok_emb = np.array(tok_emb)

In [29]:
tok_emb.shape

(110, 1024)

In [31]:
sent_emb = np.array(record_1['sentence_embeddings'])

In [32]:
sent_emb.shape

(1024,)

In [33]:
label = record_1['texts']

In [34]:
import pandas as pd

In [ ]:
data = {
    "Token Embeddings":record_1['token_embeddings'],
    "Sentence Embeddings":record_1['sentence_embeddings'],
    "Text":record_1['texts']
}

In [38]:
import numpy as np
import pandas as pd

tok_emb = np.array(chunk_list(record_1['token_embeddings'], 1024))  # (110, 1024)
sent_emb = np.array(record_1['sentence_embeddings']).reshape(1, -1)  # (1, 1024)

# Stack token + sentence embeddings together
all_emb = np.vstack([tok_emb, sent_emb])  # (111, 1024)

token_labels = [f"token_{i}" for i in range(len(tok_emb))]
sent_label = ["SENTENCE_EMB"]
all_labels = token_labels + sent_label
# Vectors TSV — no header
pd.DataFrame(all_emb).to_csv('vectors.tsv', sep="\t", index=False, header=False)

# Metadata TSV — no header either
pd.DataFrame(all_labels).to_csv('metadata.tsv', sep="\t", index=False, header=False)

In [40]:
# Build labels and categories
token_labels = [f"token_{i}" for i in range(len(tok_emb))]
token_types  = ["token"] * len(tok_emb)

sent_label = ["SENTENCE_EMB"]
sent_type  = ["sentence"]

all_labels = token_labels + sent_label
all_types  = token_types  + sent_type

# Metadata TSV — two columns, no header
meta_df = pd.DataFrame({"label": all_labels, "type": all_types})
meta_df.to_csv('metadata.tsv', sep="\t", index=False, header=True)

In [51]:
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F
text = record_1['texts']
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-0.6B")
model = AutoModel.from_pretrained("Qwen/Qwen3-0.6B")
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
inputs = {k: v for k, v in inputs.items()}

with torch.inference_mode():
    outputs = model(**inputs)

token_embeddings = outputs.last_hidden_state  # (1, seq_len, 1024)
attention_mask   = inputs["attention_mask"]

# Your mean pooling
mask_expanded       = attention_mask.unsqueeze(-1).float()
masked              = token_embeddings * mask_expanded
sentence_embedding  = masked.sum(dim=1) / mask_expanded.sum(dim=1)
sentence_embedding  = F.normalize(sentence_embedding, p=2, dim=1)

# Now compare token mean vs sentence embedding — should be ~1.0
tok_mean = token_embeddings[0].cpu().numpy().mean(axis=0)
tok_mean_normed = tok_mean / np.linalg.norm(tok_mean)
sent = sentence_embedding[0].cpu().numpy()

print(np.dot(tok_mean_normed, sent))  # ~1.0 confirms your pooling is correct

1.0


In [ ]:
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 260" width="720" height="260" role="img">
  <title>Glen Enosh — whoami terminal</title>
  <desc>Terminal block showing Glen Enosh's profile information</desc>

  <!-- Window -->
  <rect width="720" height="260" rx="10" fill="#0d1117" />

  <!-- Title bar -->
  <rect width="720" height="40" rx="10" fill="#161b22" />
  <rect y="30" width="720" height="10" fill="#161b22" />
  <rect y="39" width="720" height="1" fill="#1f2d1f" />

  <!-- Traffic lights -->
  <circle cx="20" cy="20" r="6" fill="#3a1a1a" stroke="#5c2323" stroke-width="1" />
  <circle cx="40" cy="20" r="6" fill="#2a2a1a" stroke="#4a4a23" stroke-width="1" />
  <circle cx="60" cy="20" r="6" fill="#1a3a1a" stroke="#2a6a2a" stroke-width="1" />

  <!-- Title bar text -->
  <text x="88" y="25" font-family="'Courier New', monospace" font-size="12" fill="#3a6b3a" letter-spacing="0.5">bash — ~/glen-enosh</text>

  <!-- Prompt line -->
  <text x="24" y="72" font-family="'Courier New', monospace" font-size="13">
    <tspan fill="#3fb950" font-weight="bold">glen enosh</tspan>f
    <tspan fill="#6e7681"> ~ </tspan>
    <tspan fill="#3fb950">$ </tspan>
    <tspan fill="#cdd9e5">whoami --verbose</tspan>
  </text>

  <!-- Output block bg -->
  <rect x="24" y="84" width="672" height="144" rx="0" fill="#111a11" />
  <rect x="24" y="84" width="2" height="144" fill="#2ea043" />

  <!-- handle -->
  <text x="38" y="106" font-family="'Courier New', monospace" font-size="13">
    <tspan fill="#3fb950">handle</tspan>
    <tspan fill="#6e7681">   </tspan>
    <tspan fill="#2ea043">:</tspan>
    <tspan fill="#6e7681">  </tspan>
    <tspan fill="#3fb950" font-weight="bold">@jg-eno</tspan>
    <tspan fill="#6e7681"> — NLP researcher, ML engineer, occasional caffeine-dependent optimizer</tspan>
  </text>

  <!-- now -->
  <text x="38" y="126" font-family="'Courier New', monospace" font-size="13">
    <tspan fill="#3fb950">now</tspan>
    <tspan fill="#6e7681">      </tspan>
    <tspan fill="#2ea043">:</tspan>
    <tspan fill="#6e7681">  </tspan>
    <tspan fill="#3fb950" font-weight="bold">Semantic Embedding Reconstruction</tspan>
    <tspan fill="#adbac7"> @ WSAI Lab, IIT Madras</tspan>
  </text>
  <text x="38" y="143" font-family="'Courier New', monospace" font-size="13">
    <tspan fill="#6e7681">              </tspan>
    <tspan fill="#6e7681" font-style="italic">— teaching models to unsee embeddings</tspan>
  </text>

  <!-- deep in -->
  <text x="38" y="163" font-family="'Courier New', monospace" font-size="13">
    <tspan fill="#3fb950">deep in</tspan>
    <tspan fill="#6e7681">  </tspan>
    <tspan fill="#2ea043">:</tspan>
    <tspan fill="#6e7681">  </tspan>
    <tspan fill="#adbac7">Transformer architectures · Diffusion models · </tspan>
    <tspan fill="#6e7681">KL divergence rabbit holes</tspan>
  </text>

  <!-- collab -->
  <text x="38" y="183" font-family="'Courier New', monospace" font-size="13">
    <tspan fill="#3fb950">collab</tspan>
    <tspan fill="#6e7681">   </tspan>
    <tspan fill="#2ea043">:</tspan>
    <tspan fill="#6e7681">  </tspan>
    <tspan fill="#adbac7">Research at the intersection of </tspan>
    <tspan fill="#3fb950">embeddings · generation · retrieval</tspan>
  </text>

  <!-- mail -->
  <text x="38" y="203" font-family="'Courier New', monospace" font-size="13">
    <tspan fill="#3fb950">mail</tspan>
    <tspan fill="#6e7681">     </tspan>
    <tspan fill="#2ea043">:</tspan>
    <tspan fill="#6e7681">  </tspan>
    <tspan fill="#58a6ff">jglenenosh@gmail.com</tspan>
  </text>

  <!-- pronouns -->
  <text x="38" y="223" font-family="'Courier New', monospace" font-size="13">
    <tspan fill="#3fb950">pronouns</tspan>
    <tspan fill="#2ea043">:</tspan>
    <tspan fill="#6e7681">  </tspan>
    <tspan fill="#adbac7">He / Him  · </tspan>
    <tspan fill="#6e7681" font-style="italic">Humans with AI &gt; Humans without AI</tspan>
  </text>

  <!-- Bottom prompt -->
  <text x="24" y="250" font-family="'Courier New', monospace" font-size="13">
    <tspan fill="#3fb950" font-weight="bold">glen</tspan>
    <tspan fill="#57ab5a">@</tspan>
    <tspan fill="#57ab5a" font-weight="bold">wsai-lab</tspan>
    <tspan fill="#6e7681"> ~ </tspan>
    <tspan fill="#3fb950">$ </tspan>
    <rect x="0" y="0" width="8" height="14" fill="#3fb950"/>
  </text>
  <!-- Cursor block -->
  <rect x="164" y="238" width="8" height="14" fill="#3fb950" />
</svg>